# Neural Network

In [63]:
import pandas as pd 
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split


In [72]:
df = pd.read_csv('/Users/leachen/Docs/uni/masters/stanford_mse/fall25/mse226/data_science_project/cleaned_data/train_data.csv')
X = df.drop('visits', axis=1)
y = df['visits']

categorical_cols = ['interlibrary_relation_code','fscs_definition_code', 'overdue_policy', 'beac_code', 'locale_code']
numerical_cols = ['population_lsa', 'county_population', 'print_volumes', 'ebook_volumes', 'num_lib_branches', 'num_bookmobile']


In [ ]:
label_encoders = {}
category_sizes = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    category_sizes[col] = X[col].nunique()
    print(X)

x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# X_train_num = x_train[numerical_cols]
# X_val_num = x_val[numerical_cols]
X_train_cat = x_train[categorical_cols]
X_val_cat = x_val[categorical_cols]

print(X.columns)
print(numerical_cols)

      interlibrary_relation_code                  fscs_definition_code  \
0                              0  Meets FSCS Public Library Definition   
1                              0  Meets FSCS Public Library Definition   
2                              0  Meets FSCS Public Library Definition   
3                              0  Meets FSCS Public Library Definition   
4                              0  Meets FSCS Public Library Definition   
...                          ...                                   ...   
5579                           0  Meets FSCS Public Library Definition   
5580                           0  Meets FSCS Public Library Definition   
5581                           0  Meets FSCS Public Library Definition   
5582                           0  Meets FSCS Public Library Definition   
5583                           1  Meets FSCS Public Library Definition   

                    overdue_policy        beac_code locale_code  \
0     Does not Have Overdue Policy          

KeyError: "['num_bookmobile'] not in index"

In [40]:
class NeuralNetwork(nn.Module):
    def __init__(self, category_sizes, num_numeric):
        super().__init__()
        embed_dim = 8
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_categories, embed_dim)
            for num_categories in category_sizes.values()
        ])
        total_emb_dim = embed_dim * len(category_sizes)

        self.fc = nn.Sequential(
            nn.Linear(total_emb_dim + num_numeric, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x_cat, x_num):
        embedded = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x_cat_emb = torch.cat(embedded, dim=1)
        x = torch.cat([x_cat_emb, x_num], dim=1)
        return self.fc(x)

In [42]:
wandb.init(project="mse226_neural_network",
           config = {
               "epochs": 50,
               "lr": 0.01,
               "embedding_dim": 8,
           })

config = wandb.config
model = NeuralNetwork(category_sizes, len(numerical_cols))
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = config.lr)

for epoch in range(config.epochs):
    optimizer.zero_grad()
    output = model(X_train_cat, X_train_num)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
            val_output = model(X_val_cat, X_val_num)
            val_loss = criterion(val_output, y_val)

    wandb.log({
            "fold": fold,
            "epoch": epoch,
            "train_loss": loss.item(),
            "val_loss": val_loss.item()
        })

torch.save(model.state_dict(), "model.pt")
wandb.save("model.pt")

NameError: name 'X_train_num' is not defined